# Портфолио-проект 1: E-Commerce Analytics 360°

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

В этом проекте мы проводим глубокий анализ интернет-магазина.


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect(r"d:\vibe_app\main_data_analytics\course\datasets\ecommerce.db")

# 1. SQL: Выгрузка данных для RFM-сегментации
query_rfm = """
WITH customer_orders AS (
    SELECT 
        c.customer_id,
        MAX(o.order_date) AS last_order_date,
        COUNT(DISTINCT o.order_id) AS frequency,
        ROUND(SUM(oi.total_amount), 2) AS monetary
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'Completed'
    GROUP BY c.customer_id
)
SELECT * FROM customer_orders;
"""
rfm_df = pd.read_sql(query_rfm, conn)
rfm_df['last_order_date'] = pd.to_datetime(rfm_df['last_order_date'])
now = pd.to_datetime('2026-10-01')
rfm_df['recency_days'] = (now - rfm_df['last_order_date']).dt.days

# Расчет RFM-рангов (1-4)
rfm_df['R_Score'] = pd.qcut(rfm_df['recency_days'], q=4, labels=[4, 3, 2, 1])
rfm_df['F_Score'] = pd.qcut(rfm_df['frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm_df['M_Score'] = pd.qcut(rfm_df['monetary'], q=4, labels=[1, 2, 3, 4])

print("=== RFM СЕГМЕНТАЦИЯ КЛИЕНТСКОЙ БАЗЫ ===")
print(rfm_df[['customer_id', 'recency_days', 'frequency', 'monetary', 'R_Score', 'F_Score', 'M_Score']].head(5))
